# Testmini Validation: Hugging Face Phase C Checkpoint 120

This notebook authenticates with Kaggle and Hugging Face, downloads the selected adapter repo from the Hugging Face Hub, and runs a curated `testmini` validation pack with easy, edge, and failure-prone cases using the existing inference-only code dataset.

The goal is operational validation on Kaggle T4 with clearly logged case outputs, not a full 1000-row benchmark pass.


In [ ]:
import glob, json, os, shutil, subprocess, tarfile
from pathlib import Path

WORKDIR = '/kaggle/working/RL_GSPO_Qwen2.5VLM'
VENV_DIR = '/tmp/rl-gspo-venv'
VENV_PYTHON = f'{VENV_DIR}/bin/python'
HF_REPO_ID = 'kgaero/rl-gspo-qwen2-5vlm-large-phasec-120-adapter'
HF_MODEL_DIR = '/kaggle/working/hf_phase_c_checkpoint_120'
OUTPUT_ROOT = 'outputs_testmini_casepack_hf_phase_c_120'
TARGET_LABEL = 'large_phase_c_hf_checkpoint_120'
HARDWARE_PROFILE = 'kaggle_t4'
EVAL_SPLIT = 'testmini'
CASE_PACK = 'kaggle_validation'
CASES_PER_GROUP = 2
MAX_COMPLETION_LENGTH = 192
SAVE_FULL_COMPLETION_TEXT = True
HF_TOKEN = ''
KAGGLE_API_TOKEN = ''


def find_code_dataset_root():
    matches = []
    for root, _, files in os.walk('/kaggle/input'):
        if 'reeval_bundle_manifest.json' in files and 'rl_gspo_qwen2_5vlm_eval.py' in files:
            matches.append(root)
    if not matches:
        raise FileNotFoundError(
            f"Could not find attached code dataset. Visible inputs: {glob.glob('/kaggle/input/*')}"
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Ambiguous code dataset attachment. Matches={matches}. Visible inputs: {glob.glob('/kaggle/input/*')}"
        )
    return matches[0]


CODE_DATASET_ROOT = find_code_dataset_root()
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(CODE_DATASET_ROOT, WORKDIR)

archive_path = os.path.join(WORKDIR, 'staged_rl.tar')
if os.path.exists(archive_path):
    with tarfile.open(archive_path, 'r') as tf:
        tf.extractall(WORKDIR)
    os.remove(archive_path)

nested_path = os.path.join(WORKDIR, 'staged_rl', 'staged_rl')
target_path = os.path.join(WORKDIR, 'staged_rl')
if os.path.isdir(nested_path):
    temp_path = f'{target_path}_flat'
    if os.path.exists(temp_path):
        shutil.rmtree(temp_path)
    shutil.move(nested_path, temp_path)
    shutil.rmtree(target_path)
    shutil.move(temp_path, target_path)

print('Using code dataset:', CODE_DATASET_ROOT)
print('Visible input roots:', glob.glob('/kaggle/input/*'))
print('Workdir:', WORKDIR)


In [ ]:
subprocess.run(['python3', '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['python3', '-m', 'uv', 'venv', '--seed', '--clear', VENV_DIR], check=True)

install_commands = [
    [VENV_PYTHON, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', 'pip', 'setuptools', 'wheel'],
    [
        VENV_PYTHON, '-m', 'pip', 'install', '--no-cache-dir',
        'numpy==1.26.4', 'scipy==1.15.3', 'scikit-learn==1.6.1'
    ],
    [
        VENV_PYTHON, '-m', 'pip', 'install', '--no-cache-dir',
        'pillow', 'packaging', 'safetensors', 'torchvision', 'bitsandbytes', 'xformers',
        'huggingface_hub>=0.34.0', 'datasets==4.3.0', 'transformers==4.56.2', 'trl==0.22.2',
        'unsloth', 'kaggle==2.0.0'
    ],
    [
        VENV_PYTHON, '-m', 'pip', 'install', '--no-cache-dir',
        'vllm==0.10.2', '--extra-index-url', 'https://wheels.vllm.ai/0.10.2/'
    ],
]
for command in install_commands:
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True, cwd=WORKDIR)
print('Venv ready:', VENV_PYTHON)


In [ ]:
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
access_token_path = kaggle_dir / 'access_token'
access_token_path.write_text(KAGGLE_API_TOKEN, encoding='utf-8')
os.chmod(access_token_path, 0o600)

compat_script = r'''
from huggingface_hub import whoami
from kaggle.api.kaggle_api_extended import KaggleApi
import numpy, scipy, sklearn, torch, transformers, trl, unsloth, vllm

print({
    'numpy': numpy.__version__,
    'scipy': scipy.__version__,
    'sklearn': sklearn.__version__,
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'trl': trl.__version__,
    'vllm': vllm.__version__,
})
print('HF whoami:', whoami())
api = KaggleApi()
api.authenticate()
print('Kaggle auth method:', api.get_config_value('auth_method'))
print('Kaggle username:', api.get_config_value('username'))
'''
subprocess.run([VENV_PYTHON, '-c', compat_script], check=True, cwd=WORKDIR, env=dict(os.environ))

download_script = r'''
import json, os
from huggingface_hub import snapshot_download
repo_id = os.environ['HF_REPO_ID']
local_dir = os.environ['HF_MODEL_DIR']
path = snapshot_download(repo_id=repo_id, local_dir=local_dir, token=os.environ['HF_TOKEN'])
print('Downloaded model snapshot to', path)
print('Files:', json.dumps(sorted(os.listdir(local_dir)), indent=2))
'''
env = dict(os.environ)
env['HF_REPO_ID'] = HF_REPO_ID
env['HF_MODEL_DIR'] = HF_MODEL_DIR
subprocess.run([VENV_PYTHON, '-c', download_script], check=True, cwd=WORKDIR, env=env)


In [ ]:
target_spec = {
    'targets': [
        {
            'label': TARGET_LABEL,
            'checkpoint': HF_MODEL_DIR,
            'phase': 'phase_c',
            'max_completion_length': MAX_COMPLETION_LENGTH,
        }
    ]
}
TARGET_SPEC_PATH = os.path.join(WORKDIR, f'{OUTPUT_ROOT}_targets.json')
with open(TARGET_SPEC_PATH, 'w', encoding='utf-8') as handle:
    json.dump(target_spec, handle, indent=2)
print('Target spec path:', TARGET_SPEC_PATH)
print(json.dumps(target_spec, indent=2))


In [ ]:
cmd = [
    VENV_PYTHON,
    'rl_gspo_qwen2_5vlm_eval.py',
    '--target-spec-json', TARGET_SPEC_PATH,
    '--hardware-profile', HARDWARE_PROFILE,
    '--output-root', OUTPUT_ROOT,
    '--eval-split', EVAL_SPLIT,
    '--case-pack', CASE_PACK,
    '--cases-per-group', str(CASES_PER_GROUP),
]
if SAVE_FULL_COMPLETION_TEXT:
    cmd.append('--save-full-completion-text')

env = dict(os.environ)
env['PYTHONUNBUFFERED'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

reeval_log_path = os.path.join(WORKDIR, OUTPUT_ROOT, 'reevaluation_subprocess.log')
os.makedirs(os.path.dirname(reeval_log_path), exist_ok=True)
print('Running:', ' '.join(cmd))
print('Subprocess log:', reeval_log_path)
with open(reeval_log_path, 'w', encoding='utf-8') as log_file:
    completed = subprocess.run(
        cmd,
        check=False,
        cwd=WORKDIR,
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
if completed.returncode != 0:
    print(f'Reevaluation failed with return code {completed.returncode}. Last 200 log lines:')
    with open(reeval_log_path, 'r', encoding='utf-8') as handle:
        print(''.join(handle.readlines()[-200:]))
    raise RuntimeError(f'Reevaluation subprocess failed with return code {completed.returncode}')
print('Reevaluation finished successfully.')


In [ ]:
summary_path = os.path.join(WORKDIR, OUTPUT_ROOT, 'reevaluation_summary.json')
csv_path = os.path.join(WORKDIR, OUTPUT_ROOT, 'reevaluation_summary.csv')
subset_sizes_path = os.path.join(WORKDIR, OUTPUT_ROOT, 'eval_subset_sizes.json')
target_dir = os.path.join(WORKDIR, OUTPUT_ROOT, TARGET_LABEL)
case_outputs_path = os.path.join(target_dir, 'case_outputs.json')

for path in [subset_sizes_path, summary_path, csv_path, os.path.join(target_dir, 'eval_metrics.json'), case_outputs_path]:
    print('\n===', path, '===')
    with open(path, 'r', encoding='utf-8') as handle:
        print(handle.read())


In [ ]:
import json
from pathlib import Path

target_dir = Path(WORKDIR) / OUTPUT_ROOT / TARGET_LABEL
case_outputs_path = target_dir / 'case_outputs.json'
with case_outputs_path.open('r', encoding='utf-8') as handle:
    cases = json.load(handle)

grouped = {}
for row in cases:
    grouped.setdefault(row['category'], []).append(row)

for label, rows in grouped.items():
    print('\n###', label)
    for row in rows:
        payload = {
            'pid': row.get('pid'),
            'question': row.get('question'),
            'context_family': row.get('context_family'),
            'skills': row.get('skills'),
            'precision': row.get('precision'),
            'gold_answer': row.get('gold_answer'),
            'parsed_answer': row.get('parsed_answer'),
            'normalized_exact_match': row.get('normalized_exact_match'),
            'tolerance_match': row.get('tolerance_match'),
            'failure_mode': row.get('failure_mode'),
            'completion_tokens': row.get('completion_tokens'),
            'completion': row.get('completion'),
        }
        print(json.dumps(payload, indent=2, ensure_ascii=False))
